# C1.4 · Red-teaming agents: the identity surface

**Function C — Offensive Security & Research → The Pentester / Red Teamer**  ·  *Security of AI*

Builds on **[C1.3 · Red-teaming agents: the injection surface](https://spbreed.github.io/cyber-commons/lessons/C1.3.html)**.

| | |
|---|---|
| Open-source tooling | Keycloak, SPIRE |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


The identity surface is where agentic red teaming finds the most and reports it
worst.

The findings are easy to produce, because delegation is new code and narrowing
rules are easy to get wrong. The reports are bad because they describe a clever
token manipulation instead of the **absent narrowing rule**, so the fix becomes
"block that specific request" and the class recurs next quarter.

Four attacks cover the surface, and each maps to exactly one rule from A2.5:

| Attack | Rule that should refuse it |
|---|---|
| widen scope during delegation | subset of what was presented |
| exceed the recipient's ceiling | within the actor's own ceiling |
| replay an expired token | expiry check at exchange |
| impersonate the principal | **none — this is a platform control** |

The last row is the one worth internalising: three of the four are closed by a
token format, and one is not closeable that way at all.

## 2 · Demo — build the target, then attack it

In [ ]:
import time
from dataclasses import dataclass, field

CEILINGS = {"dana@corp": {"repo:read","repo:write","deploy:prod","secrets:read"},
            "patch-agent": {"repo:read","repo:write"},
            "triage-agent": {"repo:read"},
            "deploy-agent": {"repo:read","deploy:prod"}}

class DelegationError(Exception): pass

@dataclass
class Token:
    sub: str; actor: str; scopes: set; act: dict = None
    issued: float = field(default_factory=time.time); ttl: float = 300
    @property
    def expired(self): return time.time() - self.issued > self.ttl
    def chain(self):
        out, node = [], self.act
        while node: out.append(node["actor"]); node = node.get("act")
        c = list(reversed(out)) + [self.actor]
        if c[0] != self.sub: c.insert(0, self.sub)
        return c

def mint(p):
    return Token(p, p, set(CEILINGS[p]))

def exchange(pres, actor, scopes):
    if pres.expired:
        raise DelegationError("presented token has expired")
    scopes = set(scopes)
    if not scopes <= pres.scopes:
        raise DelegationError(f"widening: {sorted(scopes - pres.scopes)} not in "
                              f"the presented token")
    if not scopes <= CEILINGS.get(actor, set()):
        raise DelegationError(f"above {actor}'s ceiling: "
                              f"{sorted(scopes - CEILINGS.get(actor, set()))}")
    return Token(pres.sub, actor, scopes, {"actor": pres.actor, "act": pres.act})

def impersonate(principal, actor, scopes):
    return Token(principal, principal, set(scopes), None)

dana  = mint("dana@corp")
patch = exchange(dana, "patch-agent", {"repo:read", "repo:write"})
print("target built. baseline chain:", " → ".join(patch.chain()))

In [ ]:
ATTACKS = [
 ("IDN-01", "widen scope during delegation", "critical",
  lambda: exchange(patch, "deploy-agent", {"deploy:prod"})),
 ("IDN-02", "exceed the recipient's ceiling", "high",
  lambda: exchange(dana, "triage-agent", {"repo:write"})),
 ("IDN-03", "replay an expired token", "high",
  lambda: exchange(Token("dana@corp", "patch-agent", {"repo:write"}, ttl=-1),
                   "deploy-agent", {"repo:read"})),
]
results = []
for aid, name, sev, fn in ATTACKS:
    try:
        fn(); through, detail = True, "SUCCEEDED"
    except DelegationError as e:
        through, detail = False, str(e)[:52]
    results.append({"id": aid, "name": name, "sev": sev,
                    "through": through, "detail": detail})

bad = impersonate("dana@corp", "patch-agent", {"repo:write"})
results.append({"id": "IDN-04", "name": "impersonate the principal",
                "sev": "critical", "through": "patch-agent" not in bad.chain(),
                "detail": f"chain is {bad.chain()} — the agent is absent"})

print(f"{'id':8s}{'severity':10s}{'through':9s}attack")
print("-" * 74)
for r in results:
    print(f"{r['id']:8s}{r['sev']:10s}{str(r['through']):9s}{r['name']}")
    print(f"{'':27s}{r['detail']}")
asr = sum(r["through"] for r in results) / len(results)
print(f"\nidentity-surface ASR: {asr:.2f}")

## 3 · Where it breaks — how this finding is usually written

The bad version of the IDN-04 report describes the token manipulation and asks the team to "validate the act claim". They will add a check, the check will look correct, and the class will recur — because the problem is not a missing validation, it is that the agent could obtain the principal's credential at all.

In [ ]:
BAD_REPORT = """
Title: Missing act claim validation
Severity: High
Detail: By omitting the `act` claim from the token, an agent can present a
        credential indistinguishable from the principal's.
Recommendation: Validate that the act claim is present on all agent requests.
"""
print(BAD_REPORT)
print("Why this gets closed and recurs:")
for why in ["it describes the payload, not the missing control",
            "'validate the act claim' is implementable and does not fix anything —",
            "  the agent still HOLDS a principal credential; it just also sends a claim",
            "no reproduction a defender can run on their own build",
            "no statement of what would prove the fix worked"]:
    print("   ·", why)

## 4 · The control — a finding report that names the missing control

In [ ]:
CONTROL_FOR_SURFACE = {
 "widen scope during delegation":  "scope narrowing at token exchange (subset of presented)",
 "exceed the recipient's ceiling": "per-actor ceiling enforced at the issuer",
 "replay an expired token":        "expiry validated at exchange AND at the resource server",
 "impersonate the principal":      "the IdP must refuse to issue a human-subject "
                                   "credential to a workload identity",
}
def finding_report(r, target="patch-agent"):
    control = CONTROL_FOR_SURFACE[r["name"]]
    return f"""[{r['id']}] {r['sev'].upper()} — {r['name']}

  Reproduction   present a token as {target} requesting the scopes above
  Observed       {r['detail']}
  Missing control
                 {control}
  NOT a fix      blocking this specific request shape. The next one differs.
  Proof of fix   the same reproduction must raise at the issuer, and a regression
                 case must fail on the current build and pass on the fixed one."""

for r in results:
    if r["through"]:
        print(finding_report(r))
        print()

In [ ]:
# Verify: prove the recommended control actually closes IDN-04.
def issue_token(subject_kind, requester_kind, allow_human_subject_to_workload):
    """The IdP control: may a workload obtain a human-subject credential?"""
    if subject_kind == "human" and requester_kind == "workload" \
            and not allow_human_subject_to_workload:
        return None, "IdP refuses: human-subject credential to a workload identity"
    return Token("dana@corp", "dana@corp", {"repo:write"}, None), "issued"

for allow in (True, False):
    tok, why = issue_token("human", "workload", allow)
    label = "current behaviour" if allow else "with the recommended control"
    print(f"{label:32s} {'ISSUED — impersonation possible' if tok else why}")
assert issue_token("human", "workload", False)[0] is None
print("\nThe reproduction now fails at the issuer, which is what 'fixed' means.")

## What you just proved

IDN-01, IDN-02 and IDN-03 are blocked, each naming the rule that refused it. IDN-04 (impersonation) succeeds, giving an identity-surface ASR of 0.25 and a chain containing only `dana@corp`. The weak report is shown alongside the structured one, and the recommended IdP control is demonstrated refusing the issuance.

## Your turn

Rewrite your last identity finding in this shape. If the "missing control" line is hard to write, the finding was about a payload — and it will be closed without fixing the class.

---

**Next → [C1.5 · Red-teaming agents: the containment surface](https://spbreed.github.io/cyber-commons/lessons/C1.5.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/C1.4.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/C1.4.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*